In [2]:
# import libraries and load environment variables
import os,json,time
from dotenv import load_dotenv
load_dotenv()
PAGEINDEX_API_KEY=os.getenv("Page_Index_API_key")
GROQ_API_KEY=os.getenv("Groq_API_Key")

print("Page Index API Key laoded ","✅"if PAGEINDEX_API_KEY else "❌")
print("Groq API Key laoded ","✅"if GROQ_API_KEY else "❌")

Page Index API Key laoded  ✅
Groq API Key laoded  ✅


In [4]:
# initialize clients
from pageindex import PageIndexClient
from groq import Groq

pi_client=PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client=Groq(api_key=GROQ_API_KEY)

print("Clients initialized ","✅"if pi_client and groq_client else "❌")

Clients initialized  ✅


In [11]:
# upload a pdf document
PDF_PATH="The_psychology_of_money.pdf"
print(f"Uploading document: {PDF_PATH} ","⏳")
result=pi_client.submit_document(PDF_PATH)
doc_id=result["doc_id"]

print(f"Document uploaded with ID: {doc_id} ","✅"if doc_id else "❌")
print(" ")

end=> Uploading document: The_psychology_of_money.pdf  ⏳
Document uploaded with ID: pi-cmq50hv7u04ve01qxt5hlvpl7  ✅
 


In [14]:
## pool until document is processed
# page index builds the tree asynchronously. we need to wait until the document is fully processed 
# for 50 page doc it'll take 1.5 minute

print("Building Tree Index ","⏳")
print("This Execute Once Per Doc and index is cached for reuse ","ℹ️")

while True:
    status_result=pi_client.get_document(doc_id)
    status=status_result["status"]
    print("Current Status: ",status)
    
    if status=="completed":
        print("Tree Index Built Successfully ","✅")
        break
    elif status=="failed":
        print("Tree Index Building Failed ","❌")
        break
    
    time.sleep(5)

Building Tree Index  ⏳
This Execute Once Per Doc and index is cached for reuse  ℹ️
Current Status:  completed
Tree Index Built Successfully  ✅


In [15]:
# inspecting the tree structure

tree_result=pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree=tree_result.get("result",[])

print(f"TopLevel Section :{len(pageindex_tree)}")
print("\n Raw Tree (first-nodes):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

TopLevel Section :2

 Raw Tree (first-nodes):
{
  "title": "Preface",
  "node_id": "0000",
  "page_index": 1,
  "summary": "The Psychology of Money\n\n![img-0.jpeg](img-0.jpeg)\n\nTIMELESS LESSONS ON WEALTH, GREED, AND HAPPINESS\n\nMORGAN HOUSEL\n\n\"House's observations often hit the daily double: they say things that haven't been said before, and they make sense.\"\n\n\u2014HOWARD MARKS\n\nH\n\nThe Psychology of Money\n\nTIMELESS LESSONS ON WEALTH, GREED, AND HAPPINESS\n\nMORGAN HOUSEL\n\nHn Harriman House\n\nOceanofPDF.com\n\nFor\nMy parents, who teach me.\nGretchen, who guides me.\nMiles and Reese, who inspire me.\n\nOceanofPDF.com\n",
  "text": "The Psychology of Money\n\n![img-0.jpeg](img-0.jpeg)\n\nTIMELESS LESSONS ON WEALTH, GREED, AND HAPPINESS\n\nMORGAN HOUSEL\n\n\"House's observations often hit the daily double: they say things that haven't been said before, and they make sense.\"\n\n\u2014HOWARD MARKS\n\nH\n\nThe Psychology of Money\n\nTIMELESS LESSONS ON WEALTH, GREED, AND

In [18]:
# Preety Print the whole Tree

def print_tree(nodes,indent=0):
    """Recursively prints the tree title for a visual representation.
    """
    
    for node in nodes:
        prefix=" " * indent + ("├── " if indent>0 else "")
        page =node.get("page_index","?")
        print(f"{prefix}[{"node_id"}]{node["title"]} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"],indent+1)
        
print("Full Document Tree Structure:")
print_tree(pageindex_tree)

Full Document Tree Structure:
[node_id]Preface (p.1)
[node_id]Introduction: The Greatest Show On Earth (p.4)
 ├── [node_id]Behavior Over Intelligence: The Financial Success Paradox (p.4)
 ├── [node_id]The Psychology of Money: Why Behavior Trumps Intelligence (p.8)
 ├── [node_id]The Psychology of Money: Why No One Is Crazy (p.12)
 ├── [node_id]The Role of Personal Experience in Financial Decision-Making (p.16)
 ├── [node_id]The Psychology of Financial Experience and Modernity (p.20)
 ├── [node_id]The Financial Learning Curve and the Role of Luck and Risk (p.24)
 ├── [node_id]The Role of Luck and Risk in Success and Failure (p.28)
 ├── [node_id]The Thin Line Between Luck, Risk, and Skill (p.32)
 ├── [node_id]The Peril of Never Having Enough (p.36)
 ├── [node_id]The Psychology of Enough and the Power of Compounding (p.40)
 ├── [node_id]The Counterintuitive Power of Compounding (p.44)
 ├── [node_id]The Power of Compounding and the Discipline of Wealth Preservation (p.48)
 ├── [node_id]The 

In [22]:
# count totel nodes in the tree
def count_nodes(nodes):
    total=len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total+=count_nodes(n["nodes"])
    return total

total=count_nodes(pageindex_tree)
print(f"\nTotal Nodes in Tree: {total}")
print("each node = one retrieval section of the document")


Total Nodes in Tree: 51
each node = one retrieval section of the document


In [28]:
# LLM Tree Function

def llm_tree_search(query:str,tree:list,model:str="llama-3.3-70b-versatile")->dict:
    """
     Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    
    """
    
    # compress  tree to save token only send tiltes + short summaries
     
    def compress(nodes):
        out=[]
        for n in nodes:
            
            entry={
                "node_id": n["node_id"],
                "title": n["title"],
                "page":n.get("page_index","?"),
                "summary":n.get("text","")[:150]
                
            }
            
            if n.get("nodes"):
                entry["children"]=compress(n["nodes"])
            out.append(entry)
        return out
    compressed_tree=compress(tree)
    
    prompt=f"""You are given a query and a document's tree structure (like a Table of Contents).
    Your task: identify which node IDs most likely contain the answer to the query.
    Think step-by-step about which sections are relevant.

    Query: {query}

    Document Tree:
    {json.dumps(compressed_tree, indent=2)}

    Reply ONLY in this exact JSON format:
    {{
    "thinking": "<your step-by-step reasoning>",
    "node_list": ["node_id1", "node_id2"]
    }}"""
    
    response=groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [30]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is pyschology of money about? "

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is pyschology of money about? 

🧠 LLM Reasoning:
The query 'What is psychology of money about?' suggests that the user is looking for an overview of the book's main topic. Given the document tree, the most relevant sections would be the introduction and any sections that summarize the book's premise. Starting with the root node '0000', which is the preface, we can see that it provides an initial overview. Then, looking at the children of node '0001', which is the introduction, we find node '0003' titled 'The Psychology of Money: Why Behavior Trumps Intelligence'. This node seems directly relevant as it likely discusses the core idea of the book. Additionally, any sections that discuss the psychology of financial decision-making, wealth, and behavior would be pertinent. Based on the titles and summaries provided, nodes '0002', '0004', and '0005' also appear to address aspects of the psychology of money, although '0003' is the most directly related. Therefore, these nodes a

In [ ]:
# find nodes by id

def find_nodes_by_ids(tree:list,target_ids:list)->list:
    """Recursively searches the tree for nodes with IDs in target_ids and returns them.
    """
    found=[]
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"],target_ids))
    return found

In [32]:
# generate answer using retrieved nodes

def generate_answer(query:str,nodes:list,model:str="llama-3.3-70b-versatile")->str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "No relevant sections found in the document."
    
    # build context string from retrieved nodes
    
    context_parts=[]
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
        context="/n/n---/n/n".join(context_parts)
        prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content
        

In [33]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [41]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What is pyschology of money about? ",
    tree=pageindex_tree
)

🔍 Query: What is pyschology of money about? 

🧠 Reasoning: The query asks for information about the book 'The Psychology of Money'. To find relevant sections, we should look for nodes that contain the title of the book or phrases related to its content. Nodes...
🎯 Retrieved node IDs: ['0000', '0001', '0003', '0004']
📄 Sections found: ['Preface']

📝 Answer:
The Psychology of Money is about "TIMELESS LESSONS ON WEALTH, GREED, AND HAPPINESS" (Preface, 1).
